In [11]:
# Day 4 - Quality Scoring System
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load saved data
metrics_df       = pd.read_csv('../data/fundamental_metrics.csv')
survival_results = pd.read_csv('../data/survival_filter_results.csv')

# Get only stocks that passed survival filter
passed_symbols   = survival_results[
    survival_results['Passed Survival'] == True
]['Symbol'].tolist()

passed_df        = metrics_df[
    metrics_df['Symbol'].isin(passed_symbols)
].copy().reset_index(drop=True)

print(f"Total stocks in metrics : {len(metrics_df)}")
print(f"Passed survival filter  : {len(passed_df)}")
print(f"\nStocks to score:")
print(passed_df['Symbol'].tolist())

Total stocks in metrics : 46
Passed survival filter  : 35

Stocks to score:
['TCS', 'RELIANCE', 'WIPRO', 'PERSISTENT', 'LTTS', 'HAPPSTMNDS', 'CHOLAFIN', 'MUTHOOTFIN', 'MANAPPURAM', 'CREDITACC', 'GARFIBRES', 'SUPRAJIT', 'HIMATSEIDE', 'HAL', 'BEL', 'MIDHANI', 'PARAS', 'VINATIORGA', 'FINEORG', 'GALAXYSURF', 'AAVAS', 'SUNPHARMA', 'DIVISLAB', 'APLLTD', 'GRANULES', 'IPCALAB', 'TITAN', 'HAVELLS', 'VGUARD', 'SYMPHONY', 'WONDERLA', 'PIIND', 'RALLIS', 'DHANUKA', 'ABSLAMC']


In [12]:
# ============================================
# SCORING HELPER FUNCTIONS
# ============================================

def score_metric_level(value, thresholds, reverse=False):
    """
    Score a metric based on absolute level
    thresholds = [excellent, good, acceptable]
    reverse = True means lower is better (e.g. Debt/Equity)
    Returns 0-3
    """
    if pd.isna(value):
        return None

    if reverse:
        # Lower is better
        if value <= thresholds[0]:   return 3
        elif value <= thresholds[1]: return 2
        elif value <= thresholds[2]: return 1
        else:                        return 0
    else:
        # Higher is better
        if value >= thresholds[0]:   return 3
        elif value >= thresholds[1]: return 2
        elif value >= thresholds[2]: return 1
        else:                        return 0


def score_metric_trend(current, previous, higher_is_better=True):
    """
    Score trend direction
    Returns 0-2
    """
    if pd.isna(current) or pd.isna(previous):
        return None
    if higher_is_better:
        if current > previous * 1.05:   return 2  # improved >5%
        elif current >= previous * 0.95: return 1  # stable
        else:                            return 0  # deteriorated
    else:
        if current < previous * 0.95:   return 2  # improved >5%
        elif current <= previous * 1.05: return 1  # stable
        else:                            return 0  # deteriorated


def safe_score(level_score, trend_score):
    """
    Combine level and trend scores safely
    Handles None values gracefully
    """
    if level_score is None and trend_score is None:
        return None
    if level_score is None:
        return trend_score
    if trend_score is None:
        return level_score
    return level_score + trend_score

print("Scoring helper functions loaded ✅")

Scoring helper functions loaded ✅


In [13]:
def calculate_historical_score(row):
    """
    Score each stock on historical trend
    Level (0-3) + Trend (0-2) = max 5 per metric
    8 metrics × 5 = 40 points total
    """
    scores  = {}
    details = {}

    is_financial = any(
        kw in str(row.get('Sector', ''))
        for kw in ['Financial', 'Banking', 'Insurance']
    )

    # ── METRIC 1: Revenue Growth (5 pts) ──
    rev_cagr = row.get('Revenue CAGR Max')
    rev_yoy  = row.get('Revenue YoY Q')

    level    = score_metric_level(rev_cagr, [25, 15, 10])
    trend    = score_metric_trend(
                   row.get('Revenue YoY Q'),
                   row.get('Revenue CAGR Max'),
                   higher_is_better=True
               )
    if trend is None: trend = 1
    scores['Revenue Growth'] = safe_score(level, trend)
    details['Revenue CAGR']  = rev_cagr
    details['Revenue YoY Q'] = rev_yoy

    # ── METRIC 2: Profit Growth (5 pts) ──
    profit_cagr = row.get('Profit CAGR Max')
    profit_yoy  = row.get('Profit YoY Q')

    level    = score_metric_level(profit_cagr, [30, 20, 10])
    trend    = score_metric_trend(
                   profit_yoy,
                   profit_cagr,
                   higher_is_better=True
               )
    if trend is None: trend = 1
    scores['Profit Growth'] = safe_score(level, trend)
    details['Profit CAGR']  = profit_cagr
    details['Profit YoY Q'] = profit_yoy

    # ── METRIC 3: Margins (5 pts) ──
    avg_opm    = row.get('Avg OPM 5Y')
    latest_opm = row.get('Latest OPM Q')

    if is_financial:
        level  = score_metric_level(avg_opm, [30, 20, 10])
    else:
        level  = score_metric_level(avg_opm, [25, 15, 10])

    trend      = score_metric_trend(
                     latest_opm,
                     avg_opm,
                     higher_is_better=True
                 )
    if trend is None: trend = 1
    scores['Margins']       = safe_score(level, trend)
    details['Avg OPM 5Y']   = avg_opm
    details['Latest OPM Q'] = latest_opm

    # ── METRIC 4: ROE (5 pts) ──
    roe     = row.get('Final ROE')
    roe_ref = row.get('ROE Avg 5Y') or row.get('Avg ROCE 5Y') or roe

    level   = score_metric_level(roe, [30, 20, 15])
    trend   = score_metric_trend(roe, roe_ref, higher_is_better=True)
    if trend is None: trend = 1
    scores['ROE']         = safe_score(level, trend)
    details['ROE']        = roe
    details['ROE Ref']    = roe_ref

    # ── METRIC 5: ROCE (5 pts) ──
    if is_financial:
        roce     = row.get('Final ROE')
        roce_ref = row.get('ROE Avg 5Y') or roce
    else:
        roce     = row.get('Latest ROCE')
        roce_ref = row.get('Avg ROCE 5Y') or roce

    level    = score_metric_level(
                   roce,
                   [35, 20, 15] if not is_financial else [20, 15, 10]
               )
    trend    = score_metric_trend(roce, roce_ref, higher_is_better=True)
    if trend is None: trend = 1
    scores['ROCE']    = safe_score(level, trend)
    details['ROCE']   = roce

    # ── METRIC 6: Debt (5 pts) ──
    debt_eq       = row.get('Debt to Equity')
    debt_reducing = row.get('Debt Reducing')

    if is_financial:
        profit_positive = row.get('Profit Positive Q', 0)
        total_quarters  = row.get('Total Quarters', 1)
        consistency     = (
            profit_positive / total_quarters * 100
        ) if total_quarters > 0 else 0
        level           = score_metric_level(consistency, [95, 85, 75])
        trend           = 2 if debt_reducing == True else 1
    else:
        level           = score_metric_level(
                              debt_eq, [0.3, 0.8, 1.5], reverse=True
                          )
        trend           = 2 if debt_reducing == True else 1 if pd.isna(
                              debt_reducing
                          ) else 0

    scores['Debt']            = safe_score(level, trend)
    details['Debt to Equity'] = debt_eq

    # ── METRIC 7: Cash Flow (5 pts) ──
    cf_years  = row.get('CF Positive Years', 0)
    cf_total  = row.get('CF Total Years',    1)
    cf_ratio  = (cf_years / cf_total * 100) if cf_total > 0 else 0
    cf_growing = row.get('CF Growing')

    level     = score_metric_level(cf_ratio, [90, 75, 60])
    trend     = 2 if cf_growing == True else 1 if pd.isna(cf_growing) else 0
    scores['Cash Flow']       = safe_score(level, trend)
    details['CF Consistency'] = round(cf_ratio, 1)

    # ── METRIC 8: EPS Growth (5 pts) ──
    eps_growth = row.get('EPS YoY Growth')
    eps_ref    = row.get('Profit CAGR Max')

    level      = score_metric_level(eps_growth, [25, 15, 5])
    trend      = score_metric_trend(
                     eps_growth,
                     eps_ref,
                     higher_is_better=True
                 )
    if trend is None: trend = 1
    scores['EPS Growth']  = safe_score(level, trend)
    details['EPS YoY']    = eps_growth

    # ── CALCULATE TOTAL ──
    valid_scores = [s for s in scores.values() if s is not None]
    if valid_scores:
        raw_total    = sum(valid_scores)
        max_possible = len(valid_scores) * 5
        total        = round(raw_total / max_possible * 40, 1)
    else:
        total        = 0

    return {
        'Symbol'               : row['Symbol'],
        'Sector'               : row.get('Sector'),
        'Historical Score'     : total,
        'Revenue Growth Score' : scores.get('Revenue Growth'),
        'Profit Growth Score'  : scores.get('Profit Growth'),
        'Margins Score'        : scores.get('Margins'),
        'ROE Score'            : scores.get('ROE'),
        'ROCE Score'           : scores.get('ROCE'),
        'Debt Score'           : scores.get('Debt'),
        'Cash Flow Score'      : scores.get('Cash Flow'),
        'EPS Growth Score'     : scores.get('EPS Growth'),
        **details
    }


# Test on TCS and a financial company
for symbol in ['TCS', 'CHOLAFIN', 'TITAN', 'PERSISTENT']:
    row   = passed_df[passed_df['Symbol'] == symbol].iloc[0]
    score = calculate_historical_score(row)
    print(f"\n{symbol}:")
    print(f"  Historical Score     : {score['Historical Score']}/40")
    print(f"  Revenue Growth Score : {score['Revenue Growth Score']}/5")
    print(f"  Profit Growth Score  : {score['Profit Growth Score']}/5")
    print(f"  Margins Score        : {score['Margins Score']}/5")
    print(f"  ROE Score            : {score['ROE Score']}/5")
    print(f"  ROCE Score           : {score['ROCE Score']}/5")
    print(f"  Debt Score           : {score['Debt Score']}/5")
    print(f"  Cash Flow Score      : {score['Cash Flow Score']}/5")
    print(f"  EPS Growth Score     : {score['EPS Growth Score']}/5")


TCS:
  Historical Score     : 22.0/40
  Revenue Growth Score : 1/5
  Profit Growth Score  : 0/5
  Margins Score        : 4/5
  ROE Score            : 4/5
  ROCE Score           : 5/5
  Debt Score           : 3/5
  Cash Flow Score      : 5/5
  EPS Growth Score     : 0/5

CHOLAFIN:
  Historical Score     : 19.0/40
  Revenue Growth Score : 2/5
  Profit Growth Score  : 2/5
  Margins Score        : 2/5
  ROE Score            : 3/5
  ROCE Score           : 4/5
  Debt Score           : 4/5
  Cash Flow Score      : 0/5
  EPS Growth Score     : 2/5

TITAN:
  Historical Score     : 23.0/40
  Revenue Growth Score : 4/5
  Profit Growth Score  : 3/5
  Margins Score        : 3/5
  ROE Score            : 4/5
  ROCE Score           : 1/5
  Debt Score           : 1/5
  Cash Flow Score      : 2/5
  EPS Growth Score     : 5/5

PERSISTENT:
  Historical Score     : 29.0/40
  Revenue Growth Score : 4/5
  Profit Growth Score  : 2/5
  Margins Score        : 4/5
  ROE Score            : 3/5
  ROCE Score      

In [14]:
# Calculate historical scores for all 35 stocks
historical_scores = []

for _, row in passed_df.iterrows():
    score = calculate_historical_score(row)
    historical_scores.append(score)

historical_df = pd.DataFrame(historical_scores)

# Sort by score
historical_df = historical_df.sort_values(
    'Historical Score', ascending=False
).reset_index(drop=True)

# Display ranking
print("=== HISTORICAL TREND RANKING ===")
print(f"{'Rank':<5} {'Symbol':<15} {'Score':>8} {'Sector'}")
print("─" * 60)
for i, row in historical_df.iterrows():
    print(f"{i+1:<5} {row['Symbol']:<15} {row['Historical Score']:>6}/40  {row['Sector']}")

=== HISTORICAL TREND RANKING ===
Rank  Symbol             Score Sector
────────────────────────────────────────────────────────────
1     PERSISTENT        29.0/40  Information Technology
2     BEL               29.0/40  Capital Goods
3     HAL               28.0/40  Capital Goods
4     SUNPHARMA         28.0/40  Healthcare
5     DHANUKA           27.0/40  Chemicals
6     MUTHOOTFIN        27.0/40  Financial Services
7     VINATIORGA        25.0/40  Chemicals
8     IPCALAB           25.0/40  Healthcare
9     SYMPHONY          24.0/40  Consumer Durables
10    GARFIBRES         24.0/40  Textiles
11    ABSLAMC           23.0/40  Financial Services
12    PARAS             23.0/40  Capital Goods
13    TITAN             23.0/40  Consumer Durables
14    GRANULES          23.0/40  Healthcare
15    TCS               22.0/40  Information Technology
16    MIDHANI           20.0/40  Capital Goods
17    PIIND             19.0/40  Chemicals
18    FINEORG           19.0/40  Chemicals
19    CHOLAFIN  

In [6]:
# ============================================
# PEER COMPARISON SCORE (40 points)
# ============================================

def calculate_peer_scores(df):
    """
    Compare each stock against sector peers
    Score based on percentile rank within sector
    """
    peer_scores = []

    # Metrics to compare with peers
    peer_metrics = {
        'Final ROE'        : True,   # higher is better
        'Revenue CAGR Max' : True,
        'Profit CAGR Max'  : True,
        'Avg OPM 5Y'       : True,
        'Latest ROCE'      : True,
        'Debt to Equity'   : False,  # lower is better
        'Revenue YoY Q'    : True,
        'Profit YoY Q'     : True,
    }

    for _, row in df.iterrows():
        symbol = row['Symbol']
        sector = row.get('Sector', 'Unknown')
        scores = {}

        # Get all stocks in same sector
        sector_peers = df[df['Sector'] == sector]

        for metric, higher_is_better in peer_metrics.items():
            stock_val = row.get(metric)

            if pd.isna(stock_val):
                scores[metric] = None
                continue

            # Get peer values for this metric
            peer_vals = sector_peers[metric].dropna()

            if len(peer_vals) < 2:
                # Only one stock in sector — give average score
                scores[metric] = 2
                continue

            # Calculate percentile rank
            if higher_is_better:
                percentile = (peer_vals < stock_val).sum() / len(peer_vals) * 100
            else:
                percentile = (peer_vals > stock_val).sum() / len(peer_vals) * 100

            # Convert percentile to score 0-5
            if percentile >= 80:   score = 5
            elif percentile >= 60: score = 4
            elif percentile >= 40: score = 3
            elif percentile >= 20: score = 2
            else:                  score = 1

            scores[metric] = score

        # Calculate total peer score
        valid = [s for s in scores.values() if s is not None]
        if valid:
            raw_total    = sum(valid)
            max_possible = len(valid) * 5
            total        = round(raw_total / max_possible * 40, 1)
        else:
            total        = 20  # neutral if no peer data

        peer_scores.append({
            'Symbol'              : symbol,
            'Sector'              : sector,
            'Peer Score'          : total,
            'Peer Count'          : len(sector_peers),
            'ROE Peer Score'      : scores.get('Final ROE'),
            'Revenue Peer Score'  : scores.get('Revenue CAGR Max'),
            'Profit Peer Score'   : scores.get('Profit CAGR Max'),
            'Margin Peer Score'   : scores.get('Avg OPM 5Y'),
            'ROCE Peer Score'     : scores.get('Latest ROCE'),
            'Debt Peer Score'     : scores.get('Debt to Equity'),
            'RevYoY Peer Score'   : scores.get('Revenue YoY Q'),
            'ProfYoY Peer Score'  : scores.get('Profit YoY Q'),
        })

    return pd.DataFrame(peer_scores).sort_values(
        'Peer Score', ascending=False
    ).reset_index(drop=True)


# Calculate peer scores
peer_df = calculate_peer_scores(passed_df)

print("=== PEER COMPARISON RANKING ===")
print(f"{'Rank':<5} {'Symbol':<15} {'Score':>8} {'Peers':>6}  {'Sector'}")
print("─" * 65)
for i, row in peer_df.iterrows():
    print(f"{i+1:<5} {row['Symbol']:<15} {row['Peer Score']:>6}/40"
          f"  {int(row['Peer Count']):>4}   {row['Sector']}")

=== PEER COMPARISON RANKING ===
Rank  Symbol             Score  Peers  Sector
─────────────────────────────────────────────────────────────────
1     PERSISTENT        32.0/40     5   Information Technology
2     DIVISLAB          30.0/40     5   Healthcare
3     GRANULES          28.0/40     5   Healthcare
4     AAVAS             27.4/40     6   Financial Services
5     MUTHOOTFIN        27.4/40     6   Financial Services
6     FINEORG           26.0/40     6   Chemicals
7     DHANUKA           26.0/40     6   Chemicals
8     ABSLAMC           26.0/40     6   Financial Services
9     VINATIORGA        26.0/40     6   Chemicals
10    TCS               25.0/40     5   Information Technology
11    SUNPHARMA         25.0/40     5   Healthcare
12    GARFIBRES         24.0/40     2   Textiles
13    BEL               24.0/40     4   Capital Goods
14    PARAS             24.0/40     4   Capital Goods
15    TITAN             24.0/40     4   Consumer Durables
16    LTTS              24.0/40    

In [15]:
# ============================================
# BUSINESS QUALITY SCORE (20 points)
# ============================================

def calculate_quality_score(row):
    """
    Score business quality based on:
    → Promoter holding and trend  (8 pts)
    → Institutional interest      (4 pts)
    → Profit consistency          (4 pts)
    → Cash flow quality           (4 pts)
    Total: 20 points
    """
    scores  = {}

    # ── PROMOTER HOLDING (4 pts) ──
    promoter = row.get('Promoter Holding')
    level    = score_metric_level(promoter, [60, 50, 40])
    scores['Promoter Holding'] = level if level is not None else 0

    # ── PROMOTER TREND (4 pts) ──
    promoter_change = row.get('Promoter Change 4Q', 0)
    if pd.isna(promoter_change):
        promoter_change = 0

    if promoter_change > 1:      pt_score = 4  # Buying more ✅
    elif promoter_change > 0:    pt_score = 3  # Slightly buying ✅
    elif promoter_change >= -1:  pt_score = 2  # Stable
    elif promoter_change >= -3:  pt_score = 1  # Slight selling ⚠️
    else:                        pt_score = 0  # Heavy selling ❌
    scores['Promoter Trend'] = pt_score

    # ── INSTITUTIONAL INTEREST (4 pts) ──
    # For multibagger hunting:
    # Low FII/DII = undiscovered = higher upside potential
    # High FII/DII = fully discovered = less upside
    fii = row.get('FII Holding', 0)
    dii = row.get('DII Holding', 0)
    if pd.isna(fii): fii = 0
    if pd.isna(dii): dii = 0
    institutional = fii + dii

    # Low institutional = undiscovered gem
    if institutional < 10:       inst_score = 4  # Hidden gem ✅
    elif institutional < 20:     inst_score = 3  # Partially discovered
    elif institutional < 35:     inst_score = 2  # Well known
    else:                        inst_score = 1  # Fully discovered
    scores['Institutional'] = inst_score

    # ── PROFIT CONSISTENCY (4 pts) ──
    profit_q     = row.get('Profit Positive Q', 0)
    total_q      = row.get('Total Quarters',    1)
    consistency  = (profit_q / total_q * 100) if total_q > 0 else 0

    if consistency >= 95:        pc_score = 4  # Perfect ✅
    elif consistency >= 85:      pc_score = 3
    elif consistency >= 75:      pc_score = 2
    else:                        pc_score = 1
    scores['Profit Consistency'] = pc_score

    # ── CASH FLOW QUALITY (4 pts) ──
    cf_years   = row.get('CF Positive Years', 0)
    cf_total   = row.get('CF Total Years',    1)
    cf_ratio   = (cf_years / cf_total * 100) if cf_total > 0 else 0
    cf_growing = row.get('CF Growing')

    if cf_ratio >= 90 and cf_growing == True:  cq_score = 4
    elif cf_ratio >= 80:                        cq_score = 3
    elif cf_ratio >= 65:                        cq_score = 2
    else:                                       cq_score = 1
    scores['CF Quality'] = cq_score

    # ── TOTAL ──
    total = sum(scores.values())

    return {
        'Symbol'                  : row['Symbol'],
        'Sector'                  : row.get('Sector'),
        'Quality Score'           : total,
        'Promoter Holding Score'  : scores['Promoter Holding'],
        'Promoter Trend Score'    : scores['Promoter Trend'],
        'Institutional Score'     : scores['Institutional'],
        'Profit Consistency Score': scores['Profit Consistency'],
        'CF Quality Score'        : scores['CF Quality'],
        'Promoter Holding %'      : promoter,
        'Promoter Change 4Q'      : promoter_change,
        'FII + DII %'             : round(institutional, 2),
    }


# Calculate for all 35 stocks
quality_scores = []
for _, row in passed_df.iterrows():
    score = calculate_quality_score(row)
    quality_scores.append(score)

quality_df = pd.DataFrame(quality_scores).sort_values(
    'Quality Score', ascending=False
).reset_index(drop=True)

print("=== BUSINESS QUALITY RANKING ===")
print(f"{'Rank':<5} {'Symbol':<15} {'Score':>8} "
      f"{'Promo%':>7} {'Instit%':>8}  {'Sector'}")
print("─" * 70)
for i, row in quality_df.iterrows():
    print(f"{i+1:<5} {row['Symbol']:<15} {row['Quality Score']:>6}/20"
          f"  {row['Promoter Holding %']:>6.1f}%"
          f"  {row['FII + DII %']:>6.1f}%"
          f"  {row['Sector']}")

=== BUSINESS QUALITY RANKING ===
Rank  Symbol             Score  Promo%  Instit%  Sector
──────────────────────────────────────────────────────────────────────
1     WIPRO               16/20    72.6%    16.6%  Information Technology
2     GARFIBRES           16/20    53.4%    19.5%  Textiles
3     LTTS                16/20    73.6%    18.8%  Information Technology
4     MIDHANI             16/20    74.0%     9.2%  Capital Goods
5     GALAXYSURF          16/20    70.9%    17.1%  Chemicals
6     FINEORG             16/20    75.0%    16.2%  Chemicals
7     VINATIORGA          16/20    74.3%    13.6%  Chemicals
8     WONDERLA            15/20    62.2%    16.7%  Consumer Services
9     TCS                 15/20    71.8%    23.2%  Information Technology
10    APLLTD              15/20    69.7%    20.4%  Healthcare
11    SYMPHONY            15/20    73.4%    14.0%  Consumer Durables
12    DHANUKA             15/20    69.7%    20.9%  Chemicals
13    ABSLAMC             15/20    74.8%    16.8%

In [8]:
# ============================================
# COMBINE ALL SCORES → FINAL RANKING
# ============================================

# Merge all three score dataframes
final_df = historical_df[['Symbol', 'Sector', 'Historical Score']].merge(
    peer_df[['Symbol', 'Peer Score']],
    on='Symbol'
).merge(
    quality_df[['Symbol', 'Quality Score',
                'Promoter Holding %', 'FII + DII %']],
    on='Symbol'
)

# Calculate final score
final_df['Final Score'] = (
    final_df['Historical Score'] +
    final_df['Peer Score'] +
    final_df['Quality Score']
).round(1)

# Sort by final score
final_df = final_df.sort_values(
    'Final Score', ascending=False
).reset_index(drop=True)

# Save
final_df.to_csv('../data/fundamental_scores.csv', index=False)

print("=== FINAL FUNDAMENTAL RANKING ===")
print(f"{'Rank':<5} {'Symbol':<15} {'Final':>7} "
      f"{'Hist':>6} {'Peer':>6} {'Qual':>6}  {'Sector'}")
print("─" * 75)
for i, row in final_df.iterrows():
    print(f"{i+1:<5} {row['Symbol']:<15}"
          f" {row['Final Score']:>6}/100"
          f" {row['Historical Score']:>5}/40"
          f" {row['Peer Score']:>5}/40"
          f" {row['Quality Score']:>5}/20"
          f"  {row['Sector']}")

=== FINAL FUNDAMENTAL RANKING ===
Rank  Symbol            Final   Hist   Peer   Qual  Sector
───────────────────────────────────────────────────────────────────────────
1     PERSISTENT        72.0/100  29.0/40  32.0/40    11/20  Information Technology
2     DHANUKA           68.0/100  27.0/40  26.0/40    15/20  Chemicals
3     VINATIORGA        67.0/100  25.0/40  26.0/40    16/20  Chemicals
4     MUTHOOTFIN        66.4/100  27.0/40  27.4/40    12/20  Financial Services
5     SUNPHARMA         66.0/100  28.0/40  25.0/40    13/20  Healthcare
6     ABSLAMC           64.0/100  23.0/40  26.0/40    15/20  Financial Services
7     BEL               64.0/100  29.0/40  24.0/40    11/20  Capital Goods
8     GARFIBRES         64.0/100  24.0/40  24.0/40    16/20  Textiles
9     GRANULES          63.0/100  23.0/40  28.0/40    12/20  Healthcare
10    TCS               62.0/100  22.0/40  25.0/40    15/20  Information Technology
11    HAL               61.0/100  28.0/40  20.0/40    13/20  Capital Goo

In [16]:
# Save all score files
historical_df.to_csv('../data/historical_scores.csv',  index=False)
peer_df.to_csv('../data/peer_scores.csv',               index=False)
quality_df.to_csv('../data/quality_scores.csv',         index=False)
final_df.to_csv('../data/fundamental_scores.csv',       index=False)

print("All score files saved ✅")
print(f"\nTop 10 Fundamental Picks:")
for i, row in final_df.head(10).iterrows():
    print(f"  {i+1}. {row['Symbol']:15} {row['Final Score']}/100  {row['Sector']}")

All score files saved ✅

Top 10 Fundamental Picks:
  1. PERSISTENT      72.0/100  Information Technology
  2. DHANUKA         68.0/100  Chemicals
  3. VINATIORGA      67.0/100  Chemicals
  4. MUTHOOTFIN      66.4/100  Financial Services
  5. SUNPHARMA       66.0/100  Healthcare
  6. ABSLAMC         64.0/100  Financial Services
  7. BEL             64.0/100  Capital Goods
  8. GARFIBRES       64.0/100  Textiles
  9. GRANULES        63.0/100  Healthcare
  10. TCS             62.0/100  Information Technology


In [17]:
# Check what quarterly data financial companies have
print("=== CHOLAFIN QUARTERLY ROWS ===")
print(all_stock_data['CHOLAFIN']['Quarterly Results'].index.tolist())

print("\n=== MUTHOOTFIN QUARTERLY ROWS ===")
print(all_stock_data['MUTHOOTFIN']['Quarterly Results'].index.tolist())

print("\n=== HDFCBANK QUARTERLY ROWS ===")
print(all_stock_data['HDFCBANK']['Quarterly Results'].index.tolist())

=== CHOLAFIN QUARTERLY ROWS ===


NameError: name 'all_stock_data' is not defined

In [18]:
import pickle
import pandas as pd

# Load saved raw data
with open('../data/raw_stock_data.pkl', 'rb') as f:
    all_stock_data = pickle.load(f)

print(f"Loaded {len(all_stock_data)} stocks ✅")

# Now check financial company quarterly rows
print("\n=== CHOLAFIN QUARTERLY ROWS ===")
print(all_stock_data['CHOLAFIN']['Quarterly Results'].index.tolist())

print("\n=== MUTHOOTFIN QUARTERLY ROWS ===")
print(all_stock_data['MUTHOOTFIN']['Quarterly Results'].index.tolist())

print("\n=== HDFCBANK QUARTERLY ROWS ===")
print(all_stock_data['HDFCBANK']['Quarterly Results'].index.tolist())

Loaded 46 stocks ✅

=== CHOLAFIN QUARTERLY ROWS ===
['Revenue', 'Interest', 'Expenses', 'Financing Profit', 'Financing Margin %', 'Other Income', 'Depreciation', 'Profit before tax', 'Tax %', 'Net Profit', 'EPS in Rs', 'Gross NPA %', 'Net NPA %']

=== MUTHOOTFIN QUARTERLY ROWS ===
['Revenue', 'Interest', 'Expenses', 'Financing Profit', 'Financing Margin %', 'Other Income', 'Depreciation', 'Profit before tax', 'Tax %', 'Net Profit', 'EPS in Rs', 'Gross NPA %', 'Net NPA %']

=== HDFCBANK QUARTERLY ROWS ===
['Revenue', 'Interest', 'Expenses', 'Financing Profit', 'Financing Margin %', 'Other Income', 'Depreciation', 'Profit before tax', 'Tax %', 'Net Profit', 'EPS in Rs', 'Gross NPA %', 'Net NPA %']


In [ ]:
# Extract NPA data for financial companies
financial_stocks = ['CHOLAFIN', 'MUTHOOTFIN', 'MANAPPURAM', 
                    'CREDITACC', 'AAVAS', 'ABSLAMC', 'HDFCBANK', 'IIFL']

print("=== NPA DATA FOR FINANCIAL STOCKS ===")
for symbol in financial_stocks:
    if symbol not in all_stock_data:
        continue
    
    qr = all_stock_data[symbol]['Quarterly Results']
    
    gross_npa = None
    net_npa   = None
    
    if 'Gross NPA %' in qr.index:
        gross_npa_series = qr.loc['Gross NPA %'].dropna()
        if len(gross_npa_series) > 0:
            gross_npa = gross_npa_series.iloc[-1]
    
    if 'Net NPA %' in qr.index:
        net_npa_series = qr.loc['Net NPA %'].dropna()
        if len(net_npa_series) > 0:
            net_npa = net_npa_series.iloc[-1]
    
    print(f"{symbol:15} Gross NPA: {gross_npa}%  Net NPA: {net_npa}%")

In [19]:
# Check raw NPA rows for CHOLAFIN
print("=== CHOLAFIN NPA RAW DATA ===")
qr = all_stock_data['CHOLAFIN']['Quarterly Results']
print(qr.loc['Gross NPA %'])
print(qr.loc['Net NPA %'])

=== CHOLAFIN NPA RAW DATA ===
Dec 2022    None
Mar 2023    None
Jun 2023    None
Sep 2023    None
Dec 2023    None
Mar 2024    None
Jun 2024    None
Sep 2024    None
Dec 2024    None
Mar 2025    None
Jun 2025    None
Sep 2025    None
Dec 2025    None
Name: Gross NPA %, dtype: object
Dec 2022    None
Mar 2023    None
Jun 2023    None
Sep 2023    None
Dec 2023    None
Mar 2024    None
Jun 2024    None
Sep 2024    None
Dec 2024    None
Mar 2025    None
Jun 2025    None
Sep 2025    None
Dec 2025    None
Name: Net NPA %, dtype: object


In [20]:
# Check if values exist as strings instead of numbers
print("=== CHOLAFIN RAW QUARTERLY TABLE ===")
qr = all_stock_data['CHOLAFIN']['Quarterly Results']
print(qr)

=== CHOLAFIN RAW QUARTERLY TABLE ===
                   Dec 2022 Mar 2023 Jun 2023 Sep 2023 Dec 2023 Mar 2024  \
Revenue              3356.0   3741.0   4100.0   4623.0   5007.0   5410.0   
Interest             1543.0   1734.0   2006.0   2204.0   2441.0   2579.0   
Expenses              912.0    902.0   1174.0   1391.0   1412.0   1417.0   
Financing Profit      901.0   1105.0    920.0   1028.0   1155.0   1414.0   
Financing Margin %     27.0     30.0     22.0     22.0     23.0     26.0   
Other Income           52.0     93.0     71.0     73.0     47.0    105.0   
Depreciation           30.0     36.0     39.0     39.0     46.0     75.0   
Profit before tax     923.0   1163.0    952.0   1062.0   1156.0   1444.0   
Tax %                  26.0     26.0     25.0     27.0     25.0     26.0   
Net Profit            685.0    855.0    710.0    773.0    872.0   1065.0   
EPS in Rs              8.33     10.4     8.64      9.4    10.39    12.68   
Gross NPA %            None     None     None     N

In [21]:
# ============================================
# FINANCIAL COMPANY SPECIFIC SCORING
# ============================================

def calculate_financial_score(row, raw_data):
    """
    Special scoring for financial companies
    Replaces standard historical score
    Uses: NIM, ROE, Loan growth, NPA, Profit growth
    """
    symbol  = row['Symbol']
    scores  = {}
    details = {}

    # Get quarterly data
    qr = raw_data.get(symbol, {}).get('Quarterly Results')

    # ── METRIC 1: Revenue (Loan Book) Growth (5 pts) ──
    rev_cagr = row.get('Revenue CAGR Max')
    rev_yoy  = row.get('Revenue YoY Q')

    level    = score_metric_level(rev_cagr, [20, 15, 10])
    trend    = score_metric_trend(rev_yoy, rev_cagr, True)
    if trend is None: trend = 1
    scores['Loan Growth']     = safe_score(level, trend)
    details['Revenue CAGR']   = rev_cagr
    details['Revenue YoY Q']  = rev_yoy

    # ── METRIC 2: Net Interest Margin (5 pts) ──
    nim      = row.get('Avg OPM 5Y')   # Financing Margin %
    nim_now  = row.get('Latest OPM Q')

    level    = score_metric_level(nim,    [25, 18, 12])
    trend    = score_metric_trend(nim_now, nim, True)
    if trend is None: trend = 1
    scores['NIM']             = safe_score(level, trend)
    details['Avg NIM 5Y']     = nim
    details['Latest NIM']     = nim_now

    # ── METRIC 3: ROE (5 pts) ──
    roe      = row.get('Final ROE')
    roe_avg  = row.get('ROE Avg 5Y') or roe

    level    = score_metric_level(roe,  [20, 15, 12])
    trend    = score_metric_trend(roe, roe_avg, True)
    if trend is None: trend = 1
    scores['ROE']             = safe_score(level, trend)
    details['ROE']            = roe

    # ── METRIC 4: Profit Growth (5 pts) ──
    profit_cagr = row.get('Profit CAGR Max')
    profit_yoy  = row.get('Profit YoY Q')

    level    = score_metric_level(profit_cagr, [25, 15, 10])
    trend    = score_metric_trend(profit_yoy, profit_cagr, True)
    if trend is None: trend = 1
    scores['Profit Growth']   = safe_score(level, trend)
    details['Profit CAGR']    = profit_cagr
    details['Profit YoY Q']   = profit_yoy

    # ── METRIC 5: EPS Growth (5 pts) ──
    eps_growth  = row.get('EPS YoY Growth')
    eps_ref     = row.get('Profit CAGR Max')

    level    = score_metric_level(eps_growth, [20, 12, 5])
    trend    = score_metric_trend(eps_growth, eps_ref, True)
    if trend is None: trend = 1
    scores['EPS Growth']      = safe_score(level, trend)
    details['EPS YoY']        = eps_growth

    # ── METRIC 6: NPA Quality (5 pts) ──
    # Get latest NPA from quarterly data
    gross_npa = None
    net_npa   = None

    if qr is not None:
        if 'Gross NPA %' in qr.index:
            npa_series = qr.loc['Gross NPA %'].dropna()
            # Filter out None values
            npa_vals   = [v for v in npa_series if v is not None]
            if npa_vals:
                gross_npa = float(npa_vals[-1])

        if 'Net NPA %' in qr.index:
            net_series = qr.loc['Net NPA %'].dropna()
            net_vals   = [v for v in net_series if v is not None]
            if net_vals:
                net_npa = float(net_vals[-1])

    if gross_npa is not None:
        # Score based on Gross NPA
        if gross_npa < 1:      npa_score = 5  # Excellent ✅
        elif gross_npa < 2:    npa_score = 4  # Good ✅
        elif gross_npa < 3:    npa_score = 3  # Acceptable
        elif gross_npa < 5:    npa_score = 2  # Watch ⚠️
        else:                  npa_score = 1  # Dangerous ❌
    else:
        npa_score = 3  # Neutral if no NPA data

    scores['NPA Quality']     = npa_score
    details['Gross NPA %']    = gross_npa
    details['Net NPA %']      = net_npa

    # ── METRIC 7: Profit Consistency (5 pts) ──
    profit_q  = row.get('Profit Positive Q', 0)
    total_q   = row.get('Total Quarters',    1)
    consistency = (profit_q / total_q * 100) if total_q > 0 else 0

    level     = score_metric_level(consistency, [95, 85, 75])
    trend     = 2  # Financial companies rarely lose money
    scores['Consistency']     = safe_score(level, trend)
    details['Profit Consisten'] = round(consistency, 1)

    # ── METRIC 8: Capital Adequacy Proxy (5 pts) ──
    # Use Equity/Revenue ratio as proxy
    # Higher equity relative to revenue = better capitalized
    equity    = row.get('Latest Equity')
    revenue   = row.get('TTM Revenue')

    if equity and revenue and revenue > 0:
        cap_ratio = equity / revenue * 100
        level     = score_metric_level(cap_ratio, [20, 12, 8])
    else:
        level     = 2  # neutral

    trend     = 2 if row.get('CF Growing') == True else 1
    scores['Capital']         = safe_score(level, trend)
    details['Capital Ratio']  = round(
        equity / revenue * 100, 2
    ) if equity and revenue and revenue > 0 else None

    # ── TOTAL ──
    valid_scores = [s for s in scores.values() if s is not None]
    if valid_scores:
        raw_total    = sum(valid_scores)
        max_possible = len(valid_scores) * 5
        total        = round(raw_total / max_possible * 40, 1)
    else:
        total        = 0

    return {
        'Symbol'              : symbol,
        'Sector'              : row.get('Sector'),
        'Historical Score'    : total,
        'Loan Growth Score'   : scores.get('Loan Growth'),
        'NIM Score'           : scores.get('NIM'),
        'ROE Score'           : scores.get('ROE'),
        'Profit Growth Score' : scores.get('Profit Growth'),
        'EPS Growth Score'    : scores.get('EPS Growth'),
        'NPA Score'           : scores.get('NPA Quality'),
        'Consistency Score'   : scores.get('Consistency'),
        'Capital Score'       : scores.get('Capital'),
        **details
    }


print("Financial scoring function loaded ✅")

Financial scoring function loaded ✅


In [22]:
# ============================================
# COMBINED HISTORICAL SCORING
# Uses financial scoring for financial cos
# Uses standard scoring for all others
# ============================================

financial_sectors = ['Financial', 'Banking', 'Insurance']

historical_scores = []

for _, row in passed_df.iterrows():
    symbol     = row['Symbol']
    sector     = str(row.get('Sector', ''))
    is_fin     = any(kw in sector for kw in financial_sectors)

    if is_fin:
        score = calculate_financial_score(row, all_stock_data)
    else:
        score = calculate_historical_score(row)

    historical_scores.append(score)
    print(f"{'FIN' if is_fin else 'STD':3} ✓ {symbol:15} "
          f"Historical Score: {score['Historical Score']}/40")

historical_df = pd.DataFrame(historical_scores).sort_values(
    'Historical Score', ascending=False
).reset_index(drop=True)

print(f"\n=== TOP 10 HISTORICAL SCORES ===")
for i, row in historical_df.head(10).iterrows():
    print(f"{i+1:2}. {row['Symbol']:15} {row['Historical Score']}/40"
          f"  {row['Sector']}")

STD ✓ TCS             Historical Score: 22.0/40
STD ✓ RELIANCE        Historical Score: 17.0/40
STD ✓ WIPRO           Historical Score: 17.0/40
STD ✓ PERSISTENT      Historical Score: 29.0/40
STD ✓ LTTS            Historical Score: 18.0/40
STD ✓ HAPPSTMNDS      Historical Score: 16.0/40
FIN ✓ CHOLAFIN        Historical Score: 26.0/40
FIN ✓ MUTHOOTFIN      Historical Score: 33.0/40
FIN ✓ MANAPPURAM      Historical Score: 18.0/40
FIN ✓ CREDITACC       Historical Score: 20.0/40
STD ✓ GARFIBRES       Historical Score: 24.0/40
STD ✓ SUPRAJIT        Historical Score: 12.0/40
STD ✓ HIMATSEIDE      Historical Score: 14.0/40
STD ✓ HAL             Historical Score: 28.0/40
STD ✓ BEL             Historical Score: 29.0/40
STD ✓ MIDHANI         Historical Score: 20.0/40
STD ✓ PARAS           Historical Score: 23.0/40
STD ✓ VINATIORGA      Historical Score: 25.0/40
STD ✓ FINEORG         Historical Score: 19.0/40
STD ✓ GALAXYSURF      Historical Score: 16.0/40
FIN ✓ AAVAS           Historical Score: 

In [23]:
# Recalculate peer and quality scores
peer_df    = calculate_peer_scores(passed_df)
quality_scores = []
for _, row in passed_df.iterrows():
    score = calculate_quality_score(row)
    quality_scores.append(score)
quality_df = pd.DataFrame(quality_scores)

# Combine all three
final_df = historical_df[['Symbol', 'Sector', 'Historical Score']].merge(
    peer_df[['Symbol', 'Peer Score']], on='Symbol'
).merge(
    quality_df[['Symbol', 'Quality Score',
                'Promoter Holding %', 'FII + DII %']], on='Symbol'
)

final_df['Final Score'] = (
    final_df['Historical Score'] +
    final_df['Peer Score'] +
    final_df['Quality Score']
).round(1)

final_df = final_df.sort_values(
    'Final Score', ascending=False
).reset_index(drop=True)

# Save
final_df.to_csv('../data/fundamental_scores.csv', index=False)
historical_df.to_csv('../data/historical_scores.csv', index=False)
peer_df.to_csv('../data/peer_scores.csv', index=False)
quality_df.to_csv('../data/quality_scores.csv', index=False)

print("=== FINAL FUNDAMENTAL RANKING (UPDATED) ===")
print(f"{'Rank':<5} {'Symbol':<15} {'Final':>7} "
      f"{'Hist':>6} {'Peer':>6} {'Qual':>6}  {'Sector'}")
print("─" * 75)
for i, row in final_df.iterrows():
    print(f"{i+1:<5} {row['Symbol']:<15}"
          f" {row['Final Score']:>6}/100"
          f" {row['Historical Score']:>5}/40"
          f" {row['Peer Score']:>5}/40"
          f" {row['Quality Score']:>5}/20"
          f"  {row['Sector']}")

=== FINAL FUNDAMENTAL RANKING (UPDATED) ===
Rank  Symbol            Final   Hist   Peer   Qual  Sector
───────────────────────────────────────────────────────────────────────────
1     MUTHOOTFIN        72.4/100  33.0/40  27.4/40    12/20  Financial Services
2     PERSISTENT        72.0/100  29.0/40  32.0/40    11/20  Information Technology
3     DHANUKA           68.0/100  27.0/40  26.0/40    15/20  Chemicals
4     VINATIORGA        67.0/100  25.0/40  26.0/40    16/20  Chemicals
5     AAVAS             66.4/100  28.0/40  27.4/40    11/20  Financial Services
6     ABSLAMC           66.0/100  25.0/40  26.0/40    15/20  Financial Services
7     SUNPHARMA         66.0/100  28.0/40  25.0/40    13/20  Healthcare
8     BEL               64.0/100  29.0/40  24.0/40    11/20  Capital Goods
9     GARFIBRES         64.0/100  24.0/40  24.0/40    16/20  Textiles
10    GRANULES          63.0/100  23.0/40  28.0/40    12/20  Healthcare
11    TCS               62.0/100  22.0/40  25.0/40    15/20  Infor

In [24]:
# CHECK — do we have 5Y growth data?
metrics_df = pd.read_csv('../data/fundamental_metrics.csv')
growth_cols = [c for c in metrics_df.columns if 'Growth' in c]
print("Growth columns available:")
for c in growth_cols:
    print(f"  {c}")

Growth columns available:
  EPS YoY Growth


In [25]:
def calculate_historical_score(row):
    """
    Historical performance score (40 pts)
    Uses 5Y CAGR as primary, falls back to 10Y or Max if unavailable
    """
    score    = 0
    details  = {}

    # ── REVENUE GROWTH (15 pts) ──
    # Use best available: 5Y → 10Y → Max
    rev_growth = (
        row.get('Revenue CAGR 5Y')  or
        row.get('Revenue CAGR 10Y') or
        row.get('Revenue CAGR Max') or 0
    )
    rev_growth = float(rev_growth or 0)

    if rev_growth >= 20:     s = 15
    elif rev_growth >= 15:   s = 12
    elif rev_growth >= 10:   s = 9
    elif rev_growth >= 5:    s = 5
    else:                    s = 0
    score += s
    details['Revenue Growth Score'] = s
    details['Revenue CAGR Used']    = round(rev_growth, 1)

    # ── PROFIT GROWTH (15 pts) ──
    profit_growth = (
        row.get('Profit CAGR 5Y')  or
        row.get('Profit CAGR 10Y') or
        row.get('Profit CAGR Max') or 0
    )
    profit_growth = float(profit_growth or 0)

    if profit_growth >= 20:  s = 15
    elif profit_growth >= 15:s = 12
    elif profit_growth >= 10:s = 9
    elif profit_growth >= 5: s = 5
    else:                    s = 0
    score += s
    details['Profit Growth Score']  = s
    details['Profit CAGR Used']     = round(profit_growth, 1)

    # ── OPERATING MARGIN (5 pts) ──
    opm = float(row.get('Avg OPM 5Y') or row.get('Avg OPM 10Y') or 0)

    if opm >= 25:            s = 5
    elif opm >= 20:          s = 4
    elif opm >= 15:          s = 3
    elif opm >= 10:          s = 2
    elif opm >= 5:           s = 1
    else:                    s = 0
    score += s
    details['OPM Score']     = s
    details['Avg OPM 5Y']    = round(opm, 1)

    # ── ROE (5 pts) ──
    roe = float(row.get('Final ROE') or row.get('ROE') or 0)

    if roe >= 25:            s = 5
    elif roe >= 20:          s = 4
    elif roe >= 15:          s = 3
    elif roe >= 10:          s = 2
    else:                    s = 0
    score += s
    details['ROE Score']     = s
    details['ROE Used']      = round(roe, 1)

    return min(score, 40), details


# ── FINANCIAL COMPANY HISTORICAL SCORE ──
def calculate_financial_score(row, raw_data=None):
    """
    Special historical score for financial companies (40 pts)
    Uses 5Y metrics where available
    """
    score   = 0
    details = {}

    # ── ROE (20 pts) — most important for financials ──
    roe = float(row.get('Final ROE') or row.get('ROE') or 0)

    if roe >= 20:            s = 20
    elif roe >= 18:          s = 17
    elif roe >= 15:          s = 13
    elif roe >= 12:          s = 8
    elif roe >= 10:          s = 4
    else:                    s = 0
    score += s
    details['ROE Score']     = s
    details['ROE']           = round(roe, 1)

    # ── PROFIT GROWTH (10 pts) ──
    profit_growth = (
        row.get('Profit CAGR 5Y')  or
        row.get('Profit CAGR 10Y') or
        row.get('Profit CAGR Max') or 0
    )
    profit_growth = float(profit_growth or 0)

    if profit_growth >= 20:  s = 10
    elif profit_growth >= 15:s = 8
    elif profit_growth >= 10:s = 6
    elif profit_growth >= 5: s = 3
    else:                    s = 0
    score += s
    details['Profit Growth Score'] = s
    details['Profit CAGR Used']    = round(profit_growth, 1)

    # ── REVENUE GROWTH (10 pts) ──
    rev_growth = (
        row.get('Revenue CAGR 5Y')  or
        row.get('Revenue CAGR 10Y') or
        row.get('Revenue CAGR Max') or 0
    )
    rev_growth = float(rev_growth or 0)

    if rev_growth >= 20:     s = 10
    elif rev_growth >= 15:   s = 8
    elif rev_growth >= 10:   s = 6
    elif rev_growth >= 5:    s = 3
    else:                    s = 0
    score += s
    details['Revenue Growth Score'] = s
    details['Revenue CAGR Used']    = round(rev_growth, 1)

    return min(score, 40), details


# ── TEST: Check what values are being used ──
print("=== GROWTH METRICS BEING USED ===\n")
print(f"{'Symbol':15} {'Rev5Y':8} {'Rev10Y':8} {'RevUsed':8} "
      f"{'Pro5Y':8} {'Pro10Y':8} {'ProUsed':8}")
print("─" * 70)

financial_sectors = ['Financial Services', 'Banking']

for _, row in metrics_df.iterrows():
    symbol = row['Symbol']

    rev5y   = float(row.get('Revenue CAGR 5Y')  or 0)
    rev10y  = float(row.get('Revenue CAGR 10Y') or 0)
    revused = rev5y or rev10y or float(row.get('Revenue CAGR Max') or 0)

    pro5y   = float(row.get('Profit CAGR 5Y')  or 0)
    pro10y  = float(row.get('Profit CAGR 10Y') or 0)
    proused = pro5y or pro10y or float(row.get('Profit CAGR Max') or 0)

    print(f"{symbol:15} {rev5y:7.1f}% {rev10y:7.1f}% {revused:7.1f}%  "
          f"{pro5y:7.1f}% {pro10y:7.1f}% {proused:7.1f}%")

=== GROWTH METRICS BEING USED ===

Symbol          Rev5Y    Rev10Y   RevUsed  Pro5Y    Pro10Y   ProUsed 
──────────────────────────────────────────────────────────────────────
TCS                10.2%    10.4%    10.2%      8.5%     9.3%     8.5%
INFY               12.4%    11.8%    12.4%     10.0%     8.0%    10.0%
HDFCBANK           22.4%    20.8%    22.4%     21.9%    21.2%    21.9%
RELIANCE           10.0%     9.9%    10.0%     15.3%    13.2%    15.3%
WIPRO               7.8%     6.6%     7.8%      6.2%     4.2%     6.2%
PERSISTENT         27.3%    20.2%    27.3%     32.7%    17.0%    32.7%
MPHASIS            10.0%     9.4%    10.0%      7.5%     9.7%     7.5%
LTTS               13.7%    15.1%    13.7%      9.0%    15.1%     9.0%
COFORGE            23.6%    17.6%    23.6%     14.9%    22.6%    14.9%
HAPPSTMNDS         24.2%     nan%    24.2%     20.8%     nan%    20.8%
CHOLAFIN           24.3%    21.4%    24.3%     32.2%    25.4%    32.2%
MUTHOOTFIN         15.9%    16.6%    15.9% 

In [30]:
def calculate_historical_score(row):
    """
    Historical performance score (40 pts)
    Uses 5Y CAGR as primary, falls back to 10Y or Max
    Applies maturity discount for short history
    """
    score   = 0
    details = {}

    # ── DATA MATURITY CHECK ──
    cagr_years = float(row.get('Revenue CAGR Years') or
                       row.get('Profit CAGR Years') or 0)

    if cagr_years < 5:
        discount = 0.50
        maturity = 'Very Short (<5Y)'
    elif cagr_years < 8:
        discount = 0.20
        maturity = 'Short (5-8Y)'
    else:
        discount = 0.00
        maturity = 'Full (8Y+)'

    details['Data Years']  = cagr_years
    details['Maturity']    = maturity
    details['Discount']    = discount

    # ── REVENUE GROWTH (15 pts) ──
    rev_growth = float(
        row.get('Revenue CAGR 5Y')  or
        row.get('Revenue CAGR 10Y') or
        row.get('Revenue CAGR Max') or 0
    )
    if rev_growth >= 20:    s = 15
    elif rev_growth >= 15:  s = 12
    elif rev_growth >= 10:  s = 9
    elif rev_growth >= 5:   s = 5
    else:                   s = 0
    score += s
    details['Revenue Growth Score'] = s
    details['Revenue CAGR Used']    = round(rev_growth, 1)

    # ── PROFIT GROWTH (15 pts) ──
    profit_growth = float(
        row.get('Profit CAGR 5Y')  or
        row.get('Profit CAGR 10Y') or
        row.get('Profit CAGR Max') or 0
    )
    if profit_growth >= 20:  s = 15
    elif profit_growth >= 15:s = 12
    elif profit_growth >= 10:s = 9
    elif profit_growth >= 5: s = 5
    else:                    s = 0
    score += s
    details['Profit Growth Score'] = s
    details['Profit CAGR Used']    = round(profit_growth, 1)

    # ── OPERATING MARGIN (5 pts) ──
    opm = float(
        row.get('Avg OPM 5Y') or
        row.get('Avg OPM 10Y') or 0
    )
    if opm >= 25:   s = 5
    elif opm >= 20: s = 4
    elif opm >= 15: s = 3
    elif opm >= 10: s = 2
    elif opm >= 5:  s = 1
    else:           s = 0
    score += s
    details['OPM Score']  = s
    details['Avg OPM 5Y'] = round(opm, 1)

    # ── ROE (5 pts) ──
    roe = float(row.get('Final ROE') or row.get('ROE') or 0)
    if roe >= 25:   s = 5
    elif roe >= 20: s = 4
    elif roe >= 15: s = 3
    elif roe >= 10: s = 2
    else:           s = 0
    score += s
    details['ROE Score'] = s
    details['ROE Used']  = round(roe, 1)

    # ── APPLY MATURITY DISCOUNT ──
    score = round(score * (1 - discount), 1)

    return min(score, 40), details


def calculate_financial_score(row, raw_data=None):
    """
    Special historical score for financial companies (40 pts)
    Uses 5Y metrics, applies maturity discount
    """
    score   = 0
    details = {}

    # ── DATA MATURITY CHECK ──
    cagr_years = float(row.get('Revenue CAGR Years') or
                       row.get('Profit CAGR Years') or 0)

    if cagr_years < 5:
        discount = 0.50
        maturity = 'Very Short (<5Y)'
    elif cagr_years < 8:
        discount = 0.20
        maturity = 'Short (5-8Y)'
    else:
        discount = 0.00
        maturity = 'Full (8Y+)'

    details['Data Years'] = cagr_years
    details['Maturity']   = maturity
    details['Discount']   = discount

    # ── ROE (20 pts) ──
    roe = float(row.get('Final ROE') or row.get('ROE') or 0)
    if roe >= 20:    s = 20
    elif roe >= 18:  s = 17
    elif roe >= 15:  s = 13
    elif roe >= 12:  s = 8
    elif roe >= 10:  s = 4
    else:            s = 0
    score += s
    details['ROE Score'] = s
    details['ROE']       = round(roe, 1)

    # ── PROFIT GROWTH (10 pts) ──
    profit_growth = float(
        row.get('Profit CAGR 5Y')  or
        row.get('Profit CAGR 10Y') or
        row.get('Profit CAGR Max') or 0
    )
    if profit_growth >= 20:  s = 10
    elif profit_growth >= 15:s = 8
    elif profit_growth >= 10:s = 6
    elif profit_growth >= 5: s = 3
    else:                    s = 0
    score += s
    details['Profit Growth Score'] = s
    details['Profit CAGR Used']    = round(profit_growth, 1)

    # ── REVENUE GROWTH (10 pts) ──
    rev_growth = float(
        row.get('Revenue CAGR 5Y')  or
        row.get('Revenue CAGR 10Y') or
        row.get('Revenue CAGR Max') or 0
    )
    if rev_growth >= 20:  s = 10
    elif rev_growth >= 15:s = 8
    elif rev_growth >= 10:s = 6
    elif rev_growth >= 5: s = 3
    else:                 s = 0
    score += s
    details['Revenue Growth Score'] = s
    details['Revenue CAGR Used']    = round(rev_growth, 1)

    # ── APPLY MATURITY DISCOUNT ──
    score = round(score * (1 - discount), 1)

    return min(score, 40), details

print("Updated functions with maturity discount ✅")

Updated functions with maturity discount ✅


In [31]:
# ── CELL 2: Re-run Historical Scores ──

financial_sectors = ['Financial Services', 'Banking']
historical_scores = []

for _, row in metrics_df.iterrows():
    symbol = row['Symbol']
    sector = row.get('Sector', '')

    if sector in financial_sectors:
        hist_score, hist_detail = calculate_financial_score(row)
    else:
        hist_score, hist_detail = calculate_historical_score(row)

    historical_scores.append({
        'Symbol'          : symbol,
        'Sector'          : sector,
        'Historical Score': hist_score,
        **hist_detail
    })

hist_df = pd.DataFrame(historical_scores)
hist_df.to_csv('../data/historical_scores.csv', index=False)

print(f"Historical scores recalculated: {len(hist_df)} stocks ✅")
print(f"\nTop 10 by historical score:")
print(hist_df[['Symbol', 'Historical Score', 
               'Revenue CAGR Used', 'Profit CAGR Used']]
      .sort_values('Historical Score', ascending=False)
      .head(10).to_string(index=False))

Historical scores recalculated: 46 stocks ✅

Top 10 by historical score:
    Symbol  Historical Score  Revenue CAGR Used  Profit CAGR Used
  CHOLAFIN              40.0               24.3              32.2
PERSISTENT              37.0               27.3              32.7
MUTHOOTFIN              34.0               15.9              11.1
     TITAN              34.0               23.5              17.4
       BEL              33.0               12.9              23.9
     PIIND              33.0               18.8              29.4
   FINEORG              31.0               16.9              20.0
   COFORGE              31.0               23.6              14.9
      IGPL              30.0               15.8              39.0
       HAL              29.0                7.6              23.7


In [32]:
# ── CELL 3: Re-run Final Scores ──
# (keep your existing peer and quality score functions unchanged)
# Just re-merge with new historical scores

# Load existing peer and quality scores
peer_df    = pd.read_csv('../data/peer_scores.csv')
quality_df = pd.read_csv('../data/quality_scores.csv')

# Merge all three components
final_df = hist_df[['Symbol', 'Sector', 'Historical Score']].merge(
    peer_df[['Symbol', 'Peer Score']], on='Symbol', how='left'
).merge(
    quality_df[['Symbol', 'Quality Score']], on='Symbol', how='left'
)

final_df['Final Score'] = (
    final_df['Historical Score'] +
    final_df['Peer Score'].fillna(0) +
    final_df['Quality Score'].fillna(0)
).round(1)

final_df.to_csv('../data/fundamental_scores.csv', index=False)

print(f"Final scores recalculated ✅")
print(f"\n=== UPDATED RANKINGS ===")
print(f"{'Rank':4} {'Symbol':15} {'Final':7} "
      f"{'Hist':6} {'Peer':6} {'Quality':8} {'Sector'}")
print("─" * 65)

for i, row in final_df.sort_values(
    'Final Score', ascending=False
).reset_index(drop=True).iterrows():
    print(f"{i+1:3}. {row['Symbol']:15} "
          f"{row['Final Score']:6.1f}  "
          f"{row['Historical Score']:5.1f}  "
          f"{row.get('Peer Score', 0):5.1f}  "
          f"{row.get('Quality Score', 0):7.1f}  "
          f"{row['Sector']}")

Final scores recalculated ✅

=== UPDATED RANKINGS ===
Rank Symbol          Final   Hist   Peer   Quality  Sector
─────────────────────────────────────────────────────────────────
  1. PERSISTENT        80.0   37.0   32.0     11.0  Information Technology
  2. MUTHOOTFIN        73.4   34.0   27.4     12.0  Financial Services
  3. FINEORG           73.0   31.0   26.0     16.0  Chemicals
  4. CHOLAFIN          70.7   40.0   21.7      9.0  Financial Services
  5. TITAN             69.0   34.0   24.0     11.0  Consumer Durables
  6. DHANUKA           69.0   28.0   26.0     15.0  Chemicals
  7. BEL               68.0   33.0   24.0     11.0  Capital Goods
  8. PIIND             68.0   33.0   23.0     12.0  Chemicals
  9. ABSLAMC           67.0   26.0   26.0     15.0  Financial Services
 10. SUNPHARMA         65.0   27.0   25.0     13.0  Healthcare
 11. HAPPSTMNDS        64.8   28.8   22.0     14.0  Information Technology
 12. GARFIBRES         64.0   24.0   24.0     16.0  Textiles
 13. TCS    

In [29]:
# Check HAPPSTMNDS and CHOLAFIN details
for symbol in ['HAPPSTMNDS', 'CHOLAFIN', 'PERSISTENT']:
    row = metrics_df[metrics_df['Symbol'] == symbol].iloc[0]
    print(f"\n{symbol}:")
    print(f"  Revenue CAGR 5Y  : {row.get('Revenue CAGR 5Y')}")
    print(f"  Revenue CAGR 10Y : {row.get('Revenue CAGR 10Y')}")
    print(f"  Revenue CAGR Max : {row.get('Revenue CAGR Max')}")
    print(f"  Revenue CAGR Yrs : {row.get('Revenue CAGR Years')}")
    print(f"  Profit CAGR 5Y   : {row.get('Profit CAGR 5Y')}")
    print(f"  Profit CAGR 10Y  : {row.get('Profit CAGR 10Y')}")
    print(f"  Profit CAGR Max  : {row.get('Profit CAGR Max')}")
    print(f"  Profit CAGR Yrs  : {row.get('Profit CAGR Years')}")
    print(f"  Avg OPM 5Y       : {row.get('Avg OPM 5Y')}")
    print(f"  Final ROE        : {row.get('Final ROE')}")


HAPPSTMNDS:
  Revenue CAGR 5Y  : 24.18
  Revenue CAGR 10Y : nan
  Revenue CAGR Max : 23.18
  Revenue CAGR Yrs : 6.0
  Profit CAGR 5Y   : 20.77
  Profit CAGR 10Y  : nan
  Profit CAGR Max  : 53.76
  Profit CAGR Yrs  : 6
  Avg OPM 5Y       : 22.4
  Final ROE        : 11.36

CHOLAFIN:
  Revenue CAGR 5Y  : 24.34
  Revenue CAGR 10Y : 21.43
  Revenue CAGR Max : 21.43
  Revenue CAGR Yrs : 10.0
  Profit CAGR 5Y   : 32.24
  Profit CAGR 10Y  : 25.38
  Profit CAGR Max  : 25.38
  Profit CAGR Yrs  : 10
  Avg OPM 5Y       : 24.8
  Final ROE        : 20.0

PERSISTENT:
  Revenue CAGR 5Y  : 27.34
  Revenue CAGR 10Y : 20.23
  Revenue CAGR Max : 20.23
  Revenue CAGR Yrs : 10.0
  Profit CAGR 5Y   : 32.72
  Profit CAGR 10Y  : 17.01
  Profit CAGR Max  : 17.01
  Profit CAGR Yrs  : 10
  Avg OPM 5Y       : 17.0
  Final ROE        : 24.22


In [33]:
# Save updated files
print("Saving updated scores...")

hist_df.to_csv('../data/historical_scores.csv', index=False)
final_df.to_csv('../data/fundamental_scores.csv', index=False)

# Verify only 35 stocks saved (exclude stocks without peer/quality scores)
final_35 = final_df.dropna(subset=['Peer Score', 'Quality Score'])
final_35.to_csv('../data/fundamental_scores.csv', index=False)

print(f"Saved {len(final_35)} stocks to fundamental_scores.csv ✅")
print(f"\nTop 10 final:")
for i, row in final_35.sort_values(
    'Final Score', ascending=False
).head(10).reset_index(drop=True).iterrows():
    print(f"  {i+1:2}. {row['Symbol']:15} {row['Final Score']}/100")

Saving updated scores...
Saved 35 stocks to fundamental_scores.csv ✅

Top 10 final:
   1. PERSISTENT      80.0/100
   2. MUTHOOTFIN      73.4/100
   3. FINEORG         73.0/100
   4. CHOLAFIN        70.7/100
   5. DHANUKA         69.0/100
   6. TITAN           69.0/100
   7. BEL             68.0/100
   8. PIIND           68.0/100
   9. ABSLAMC         67.0/100
  10. SUNPHARMA       65.0/100
